In [0]:
from pyspark.sql import functions as F

df_est = spark.table("workspace.default.silver_estabelecimentos")
df_emp = spark.table("workspace.default.silver_empresas")
df_mun = spark.table("workspace.default.silver_dominio_municipios")
df_cna = spark.table("workspace.default.silver_dominio_cnaes")

df_mercado_cnae = (
    df_est.alias("est")
    .join(df_emp.alias("emp"), "nr_cnpj_basico", "inner")
    .join(df_mun.alias("mun"), F.col("est.cd_municipio") == F.col("mun.cd_municipio"), "left")
    .join(df_cna.alias("cna"), F.col("est.cd_cnae_fiscal_principal") == F.col("cna.cd_cnae"), "left")
    .groupBy(
        F.col("est.sg_uf"),
        F.col("est.cd_municipio"),
        F.col("mun.ds_municipio").alias("nm_municipio"),
        F.col("est.cd_cnae_fiscal_principal"),
        F.col("cna.ds_cnae").alias("ds_cnae_fiscal_principal"),
        F.year(F.col("est.dt_inicio_atividade")).alias("nu_ano_abertura")
    )
    .agg(
        F.count("est.nr_cnpj_completo").alias("qt_total_estabelecimentos"),
        F.sum(F.when(F.col("est.cd_situacao_cadastral") == "02", 1).otherwise(0)).alias("qt_estabelecimentos_ativos"),
        F.sum(F.when(F.col("est.cd_situacao_cadastral").isin("04", "08"), 1).otherwise(0)).alias("qt_estabelecimentos_baixados_inaptos"),
        F.sum(F.col("emp.vl_capital_social")).alias("vl_capital_social_acumulado"),
        F.avg(F.col("emp.vl_capital_social")).cast("decimal(18,2)").alias("vl_capital_social_medio")
    )
    .withColumn(
        "tx_sobrevivencia_setorial",
        F.round((F.col("qt_estabelecimentos_ativos") / F.col("qt_total_estabelecimentos")) * 100, 2)
    )
    .withColumn("dt_carga_gold", F.current_timestamp())
)

(
    df_mercado_cnae.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sg_uf")
    .saveAsTable("workspace.default.gold_dp_inteligencia_mercadologica_cnae")
)
print("-> workspace.default.gold_dp_inteligencia_mercadologica_cnae gravada com sucesso.")

-> workspace.default.gold_dp_inteligencia_mercadologica_cnae gravada com sucesso.
